In [9]:
# ================================================================
# CELL 1 — IMPORTS: LOCAL SCRATCH LLM
# ================================================================
import os, cv2, json, math, time, random, warnings, re, hashlib
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from collections import Counter
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    TORCH_IMPORT_ERROR = e

try:
    from skimage.feature import graycomatrix, graycoprops
    SKIMAGE_OK = True
except ImportError:
    SKIMAGE_OK = False
    print('skimage not found — GLCM features disabled')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_OK:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print('=' * 72)
print('  STEEL DEFECT DETECTION — BiLSTM LLM')
print('=' * 72)
print('The classifier is a from-scratch BiLSTM trained locally on the SteelDefectX dataset.')
print('PyTorch available:', TORCH_OK)
if not TORCH_OK:
    print('Install PyTorch first: pip install torch torchvision torchaudio')
print('=' * 72)

  STEEL DEFECT DETECTION — BiLSTM LLM
The classifier is a from-scratch BiLSTM trained locally on the SteelDefectX dataset.
PyTorch available: True


In [10]:
# ==========================================================
# CELL 2 - CONFIGURATION  (BiLSTM only, SteelDefectX - ALL classes)
# ==========================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    torch.manual_seed(SEED)
except Exception:
    pass

# --- paths: SteelDefectX images, one folder per defect class ---
BASE_PATH  = r'C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\SteelDefectX'
IMAGES_DIR = os.path.join(BASE_PATH, 'train_by_class')
OUTPUT_DIR = os.path.join(BASE_PATH, 'OUTPUT_BILSTM')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- ALL defect classes, auto-discovered from the folder names ---
CLASS_NAMES = sorted([d for d in os.listdir(IMAGES_DIR)
                      if os.path.isdir(os.path.join(IMAGES_DIR, d))])
LABEL_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
ID_TO_LABEL = {i: c for c, i in LABEL_TO_ID.items()}

IMG_SIZE    = 200
TRAIN_SPLIT = 0.80

# --- BiLSTM (SteelSense-BiLSTM) hyper-parameters ---
MODEL_NAME_2     = 'SteelSense-BiLSTM'
BATCH_SIZE       = 32
MAX_LEN          = 160
EMBED_DIM        = 96
MIN_TOKEN_FREQ   = 1
HIDDEN_DIM_2     = 192
NUM_LAYERS_2     = 2
DROPOUT_2        = 0.30
EPOCHS_2         = 30
LEARNING_RATE_2  = 8e-4
LABEL_SMOOTH_2   = 0.06
AUGMENT_LLM2     = 2       # extra jittered prompt variants per training image
JITTER_SCALE_2   = 0.04    # magnitude of the augmentation jitter
ENSEMBLE_SIZE_2  = 5       # snapshot ensemble: top-K val checkpoints averaged
MODEL_PATH_2     = os.path.join(OUTPUT_DIR, 'steeldefectx_bilstm.pt')
TOKENIZER_PATH_2 = os.path.join(OUTPUT_DIR, 'steeldefectx_bilstm_tokenizer.json')

# --- one distinct colour per class (from the tab20 colormap) ---
def _hex(rgba):
    return '#%02x%02x%02x' % tuple(int(255 * x) for x in rgba[:3])
DEFECT_COLORS = {c: _hex(plt.cm.tab20(i % 20)) for i, c in enumerate(CLASS_NAMES)}


def collect_image_tasks():
    """One task per image: {'cls','fname','path'} for every class folder."""
    tasks = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(IMAGES_DIR, cls)
        if not os.path.isdir(cls_dir):
            print(f'Missing dir: {cls_dir}')
            continue
        for fname in sorted(os.listdir(cls_dir)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                tasks.append({'cls': cls, 'fname': fname, 'path': os.path.join(cls_dir, fname)})
    return tasks


# --- extra config for the detection + visualization pipeline ---
ANNOTATIONS_DIR = os.path.join(BASE_PATH, '_NO_XML_ANNOTATIONS')  # none -> contour fallback
RESULTS_JSON_2  = os.path.join(OUTPUT_DIR, 'steeldefectx_bilstm_runall.json')
PRINT_EVERY     = 200
SAVE_EVERY      = 500
SOURCE_COLORS = {
    'xml_annotation':     '#00FF88',
    'contour_detection':  '#4FC3F7',
    'full_image_fallback':'#EF5350',
    'scratch_tiny_llm':   '#FFA726',
}

def normalize_label(s):   # SteelDefectX has no XML; passthrough kept for safety
    return s


print(f'Classes ({len(CLASS_NAMES)}) : {CLASS_NAMES}')
print(f'IMAGES_DIR : {IMAGES_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')


Classes (24) : ['Bright scratch', 'Crazing', 'Crease', 'Crescent gap', 'Dark scratches', 'Finishing roll printing', 'Inclusion', 'Iron scale compression', 'Iron sheet ash', 'Oil spot', 'Oxide scale of plate system', 'Oxide scale of temperature system', 'Patches', 'Pitted surface', 'Punching', 'Red iron sheet', 'Rolled in scale', 'Rolled pit', 'Secondary rust skin', 'Silk spot', 'Slag inclusion', 'Waist folding', 'Welding line', 'White rust']
IMAGES_DIR : C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\SteelDefectX\train_by_class
OUTPUT_DIR : C:\Users\anmol\OneDrive\Desktop\Steel_Surface_Defect_NEU_DET-DATASET\SteelDefectX\OUTPUT_BILSTM


In [11]:
# ================================================================
# CELL 3 — RICH FEATURE EXTRACTION
# ================================================================

def compute_glcm(gray_u8):
    if not SKIMAGE_OK:
        return {}
    try:
        g    = (gray_u8 // 32).astype(np.uint8)
        glcm = graycomatrix(g, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4],
                            levels=8, symmetric=True, normed=True)
        return {
            'glcm_contrast':    round(float(np.mean(graycoprops(glcm,'contrast'))),    3),
            'glcm_homogeneity': round(float(np.mean(graycoprops(glcm,'homogeneity'))), 3),
            'glcm_energy':      round(float(np.mean(graycoprops(glcm,'energy'))),       3),
            'glcm_correlation': round(float(np.mean(graycoprops(glcm,'correlation'))),  3),
        }
    except Exception:
        return {}


def extract_rich_features(img_path, img_size=200):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None, None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res = cv2.resize(img_rgb, (img_size, img_size))
    gray    = cv2.cvtColor(img_res, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gray_u8 = gray.astype(np.uint8)
    H, W    = gray.shape
    f = {}

    # Basic stats
    f['mean_intensity'] = round(float(np.mean(gray)), 2)
    f['std_intensity']  = round(float(np.std(gray)),  2)
    f['contrast']       = round(float(np.std(gray)/(np.mean(gray)+1e-5)), 3)
    p = np.histogram(gray_u8, bins=256, range=(0,256))[0].astype(float)
    p /= p.sum() + 1e-9
    p = p[p > 0]
    f['entropy'] = round(float(-np.sum(p*np.log2(p))), 3)

    # Edges
    edges = cv2.Canny(gray_u8, 50, 150)
    # Fix for OpenCV 4.8.0 crash with float32 input
    gx = cv2.Sobel(gray_u8, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_u8, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    f['edge_density']    = round(float(np.sum(edges>0)/edges.size*100), 2)
    f['sobel_magnitude'] = round(float(np.mean(mag)), 3)
    f['laplacian_var']   = round(float(np.var(cv2.Laplacian(gray_u8,cv2.CV_64F))), 2)
    f['angle_variance']  = round(float(np.var(np.arctan2(gy, gx))), 3)
    h_e = np.sum(np.abs(gy)); v_e = np.sum(np.abs(gx)); tot = h_e+v_e+1e-5
    f['horizontal_edge_ratio'] = round(float(h_e/tot), 3)
    f['vertical_edge_ratio']   = round(float(v_e/tot), 3)

    # Morphological regions
    _, binary   = cv2.threshold(gray_u8, 0, 255, cv2.THRESH_OTSU)
    inv         = cv2.bitwise_not(binary)
    morphed     = cv2.morphologyEx(inv, cv2.MORPH_CLOSE, np.ones((3,3),np.uint8))
    contours, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    areas, circs, aspects, bboxes_raw = [], [], [], []
    for c in contours:
        a = cv2.contourArea(c)
        if a < 30: continue
        areas.append(a)
        p_len = cv2.arcLength(c, True)
        circs.append((4*np.pi*a)/(p_len**2) if p_len>0 else 0)
        x,y,bw,bh = cv2.boundingRect(c)
        bboxes_raw.append((x,y,bw,bh))
        aspects.append(max(bw,bh)/(min(bw,bh)+1e-5))
    n = len(areas)
    f['num_regions']      = n
    f['avg_area']         = round(float(np.mean(areas))  if n>0 else 0, 2)
    f['max_area']         = round(float(np.max(areas))   if n>0 else 0, 2)
    f['total_defect_pct'] = round(float(np.sum(areas)/(H*W)*100) if n>0 else 0, 2)
    f['avg_circularity']  = round(float(np.mean(circs))  if n>0 else 0, 3)
    f['min_circularity']  = round(float(np.min(circs))   if n>0 else 0, 3)
    f['avg_aspect_ratio'] = round(float(np.mean(aspects)) if n>0 else 1, 2)
    f['max_aspect_ratio'] = round(float(np.max(aspects))  if n>0 else 1, 2)
    bins = [0,0,0,0]
    for a in areas:
        if   a < 100:  bins[0]+=1
        elif a < 500:  bins[1]+=1
        elif a < 3000: bins[2]+=1
        else:          bins[3]+=1
    f['regions_tiny']=bins[0]; f['regions_small']=bins[1]
    f['regions_medium']=bins[2]; f['regions_large']=bins[3]

    # Quadrant
    q1,q2=gray[:H//2,:W//2],gray[:H//2,W//2:]
    q3,q4=gray[H//2:,:W//2],gray[H//2:,W//2:]
    qs={'top_left':np.std(q1),'top_right':np.std(q2),'bottom_left':np.std(q3),'bottom_right':np.std(q4)}
    qa=H*W/4
    qe={'top_left':np.sum(edges[:H//2,:W//2]>0)/qa*100,'top_right':np.sum(edges[:H//2,W//2:]>0)/qa*100,
        'bottom_left':np.sum(edges[H//2:,:W//2]>0)/qa*100,'bottom_right':np.sum(edges[H//2:,W//2:]>0)/qa*100}
    f['quadrant_std']={k:round(float(v),2) for k,v in qs.items()}
    f['quadrant_edge_density']={k:round(float(v),2) for k,v in qe.items()}
    f['most_active_quadrant']=max(qs,key=qs.get)
    f['edge_row_peak_pct']=round(float(np.argmax(np.sum(edges,axis=1)))/H*100,1)
    f['edge_col_peak_pct']=round(float(np.argmax(np.sum(edges,axis=0)))/W*100,1)

    # Brightness
    f['dark_pixel_pct']   = round(float(np.sum(gray<80)/gray.size*100),2)
    f['bright_pixel_pct'] = round(float(np.sum(gray>180)/gray.size*100),2)
    f['mid_pixel_pct']    = round(100-f['dark_pixel_pct']-f['bright_pixel_pct'],2)

    # FFT
    fft_mag = np.abs(np.fft.fftshift(np.fft.fft2(gray)))
    cy,cx   = H//2,W//2
    f['fft_center_energy']=round(float(np.sum(fft_mag[cy-20:cy+20,cx-20:cx+20])/(fft_mag.sum()+1e-9)),4)

    f.update(compute_glcm(gray_u8))
    f['_raw_bboxes'] = bboxes_raw
    return f, img_res

print('Cell 3 OK — extract_rich_features() defined')

Cell 3 OK — extract_rich_features() defined


In [12]:
# ================================================================
# CELL 4 — TOKENIZER, DATASET & SHARED TRAINING HELPERS
# ================================================================

if not TORCH_OK:
    raise ImportError('PyTorch is required for the scratch tiny LLM. Install: pip install torch torchvision torchaudio')

SPECIAL_TOKENS = ['<pad>', '<unk>', '<cls>']
PAD_ID, UNK_ID, CLS_ID = 0, 1, 2

class SimpleTokenizer:
    def __init__(self, token_to_id=None, max_len=160):
        self.token_to_id = token_to_id or {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
        self.max_len = max_len

    def tokenize(self, text):
        return re.findall(r'[a-zA-Z0-9_\.\-]+', str(text).lower())

    def build(self, texts, min_freq=1):
        counter = Counter()
        for text in texts:
            counter.update(self.tokenize(text))
        for tok, freq in sorted(counter.items()):
            if freq >= min_freq and tok not in self.token_to_id:
                self.token_to_id[tok] = len(self.token_to_id)
        return self

    def encode(self, text):
        ids = [CLS_ID] + [self.token_to_id.get(tok, UNK_ID) for tok in self.tokenize(text)]
        ids = ids[:self.max_len]
        attn = [1] * len(ids)
        while len(ids) < self.max_len:
            ids.append(PAD_ID)
            attn.append(0)
        return ids, attn

    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'token_to_id': self.token_to_id, 'max_len': self.max_len}, f, indent=2)

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            obj = json.load(f)
        return cls(obj['token_to_id'], obj.get('max_len', MAX_LEN))


class SteelPromptDataset(Dataset):
    def __init__(self, rows, tokenizer):
        self.rows = rows
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        text, label = self.rows[idx]
        ids, attn = self.tokenizer.encode(text)
        return {
            'input_ids': torch.tensor(ids, dtype=torch.long),
            'attention_mask': torch.tensor(attn, dtype=torch.bool),
            'label': torch.tensor(label, dtype=torch.long)
        }


SCRATCH_LLM_MODEL = None
SCRATCH_TOKENIZER = None
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def stratified_split(rows, train_ratio=0.80):
    by_label = {i: [] for i in range(len(CLASS_NAMES))}
    for row in rows:
        by_label[row[1]].append(row)
    train_rows, val_rows = [], []
    rnd = random.Random(SEED)
    for label, items in by_label.items():
        rnd.shuffle(items)
        cut = max(1, int(len(items) * train_ratio))
        train_rows.extend(items[:cut])
        val_rows.extend(items[cut:])
    rnd.shuffle(train_rows)
    rnd.shuffle(val_rows)
    return train_rows, val_rows


def evaluate_loader(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    per_class = {c: {'correct': 0, 'total': 0} for c in CLASS_NAMES}
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            y = batch['label'].to(DEVICE)
            logits = model(ids, mask)
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=1)
            loss_sum += float(loss.item()) * y.size(0)
            total += y.size(0)
            correct += int((pred == y).sum().item())
            for yy, pp in zip(y.cpu().tolist(), pred.cpu().tolist()):
                cls = ID_TO_LABEL[yy]
                per_class[cls]['total'] += 1
                if yy == pp:
                    per_class[cls]['correct'] += 1
    return correct / max(total, 1), loss_sum / max(total, 1), per_class


print('Cell 5 OK — scratch tiny LLM classifier defined')

Cell 5 OK — scratch tiny LLM classifier defined


In [13]:
# ================================================================
# CELL 5B — SECOND FROM-SCRATCH LLM: BiLSTM + RICH PROMPT + ENSEMBLE
# A different architecture family from the Transformer in Cell 5,
# trained independently from random init, producing its own results.
#
# Extra accuracy levers stacked on top of the multi-view-pooling BiLSTM:
#   1) features_to_prompt_llm2()  -- a much richer structured-text prompt
#      (finer bins + GLCM + per-quadrant stats + spectral/edge-peak info)
#      so confusable classes (scratches vs patches vs crazing) leave a
#      more distinctive token signature for the model to learn from.
#   2) build_training_rows_llm2() -- light feature-space jitter creates
#      AUGMENT_LLM2 extra training variants per image (more data for a
#      from-scratch model without touching the original dataset).
#   3) snapshot ensembling -- the top ENSEMBLE_SIZE_2 checkpoints (by val
#      accuracy) are kept and their softmax probabilities are averaged at
#      inference time, which smooths out single-run variance.
# ================================================================

class SteelSense_BiLSTM(nn.Module):
    """Second from-scratch tiny LLM.
    Token embeddings -> bidirectional LSTM -> three pooled views
    (learned attention pooling, masked max-pooling, masked mean-pooling)
    concatenated -> classification head. No pretrained weights or
    external API; trained from random initialization, with a recurrent
    encoder instead of self-attention.
    """
    def __init__(self, vocab_size, num_classes, hidden_dim=192, embed_dim=96, layers=2, dropout=0.30):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.embed_dropout = nn.Dropout(dropout * 0.5)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        enc_dim = hidden_dim * 2
        self.attn_score = nn.Sequential(
            nn.Linear(enc_dim, enc_dim // 2),
            nn.Tanh(),
            nn.Linear(enc_dim // 2, 1)
        )
        self.norm = nn.LayerNorm(enc_dim * 3)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(enc_dim * 3, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        mask = attention_mask.bool()
        x = self.embed_dropout(self.token_embedding(input_ids))
        out, _ = self.lstm(x)                                    # (B, T, 2H)

        scores = self.attn_score(out).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-inf'))
        attn_w = torch.softmax(scores, dim=1).unsqueeze(-1)
        attn_pool = torch.sum(out * attn_w, dim=1)

        masked_out = out.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        max_pool, _ = masked_out.max(dim=1)

        mask_f = mask.unsqueeze(-1).float()
        mean_pool = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)

        pooled = self.norm(torch.cat([attn_pool, max_pool, mean_pool], dim=-1))
        return self.classifier(pooled)


def _fbin(value, edges, names):
    for name, hi in zip(names, edges):
        if value < hi:
            return name
    return names[-1]


def features_to_prompt_llm2(f):
    """Richer structured-text prompt for the second LLM.
    Reuses the same rich feature dict as LLM 1 but keeps far more of it
    (GLCM texture stats, per-quadrant std/edge density, spectral energy,
    edge-peak locations) and bins continuous values more finely, giving
    the BiLSTM more signal to separate visually-similar defect classes.
    """
    b6 = ['_b0', '_b1', '_b2', '_b3', '_b4', '_b5', '_b6']
    entropy_bin  = 'entropy'  + _fbin(float(f.get('entropy', 0)),         [3.5, 4.5, 5.2, 5.8, 6.4, 7.0, 99],   b6)
    area_bin     = 'area'     + _fbin(float(f.get('avg_area', 0)),        [50, 150, 350, 800, 2000, 5000, 1e9], b6)
    aspect_bin   = 'aspect'   + _fbin(float(f.get('max_aspect_ratio', 1)),[1.4, 2.0, 2.8, 4.0, 6.0, 9.0, 1e9],  b6)
    coverage_bin = 'coverage' + _fbin(float(f.get('total_defect_pct', 0)),[2, 5, 9, 15, 24, 35, 100],           b6)
    dark_bin     = 'dark'     + _fbin(float(f.get('dark_pixel_pct', 0)),  [4, 9, 16, 25, 36, 50, 100],          b6)
    bright_bin   = 'bright'   + _fbin(float(f.get('bright_pixel_pct', 0)),[2, 6, 12, 20, 30, 45, 100],          b6)
    edge_bin     = 'edge'     + _fbin(float(f.get('edge_density', 0)),    [2, 4, 7, 11, 16, 24, 100],           b6)
    sobel_bin    = 'sobel'    + _fbin(float(f.get('sobel_magnitude', 0)), [10, 20, 35, 55, 80, 120, 1e9],       b6)
    lap_bin      = 'lap'      + _fbin(float(f.get('laplacian_var', 0)),   [50, 150, 350, 700, 1300, 2500, 1e9], b6)
    fft_bin      = 'fft'      + _fbin(float(f.get('fft_center_energy', 0)),[0.05, 0.1, 0.18, 0.28, 0.4, 0.55, 1.0], b6)

    qstd  = f.get('quadrant_std', {})
    qedge = f.get('quadrant_edge_density', {})
    quad_tokens = []
    for q in ('top_left', 'top_right', 'bottom_left', 'bottom_right'):
        quad_tokens.append(f"qstd_{q}_{int(round(float(qstd.get(q, 0))))}")
        quad_tokens.append(f"qedge_{q}_{int(round(float(qedge.get(q, 0))))}")

    tokens = [
        'steel', 'surface', 'defect', 'classification',
        f"regions_{int(f.get('num_regions', 0))}",
        f"tiny_{int(f.get('regions_tiny', 0))}",
        f"small_{int(f.get('regions_small', 0))}",
        f"medium_{int(f.get('regions_medium', 0))}",
        f"large_{int(f.get('regions_large', 0))}",
        entropy_bin, area_bin, aspect_bin, coverage_bin,
        dark_bin, bright_bin, edge_bin, sobel_bin, lap_bin, fft_bin,
        f"active_{f.get('most_active_quadrant', 'center')}",
        f"h_edge_{round(float(f.get('horizontal_edge_ratio', 0)), 2)}",
        f"v_edge_{round(float(f.get('vertical_edge_ratio', 0)), 2)}",
        f"row_peak_{int(round(float(f.get('edge_row_peak_pct', 0)) / 10) * 10)}",
        f"col_peak_{int(round(float(f.get('edge_col_peak_pct', 0)) / 10) * 10)}",
        f"circularity_{round(float(f.get('avg_circularity', 0)), 2)}",
        f"min_circularity_{round(float(f.get('min_circularity', 0)), 2)}",
        f"aspect_avg_{round(float(f.get('avg_aspect_ratio', 1)), 1)}",
        f"glcm_contrast_{round(float(f.get('glcm_contrast', 0)), 2)}",
        f"glcm_homogeneity_{round(float(f.get('glcm_homogeneity', 0)), 2)}",
        f"glcm_energy_{round(float(f.get('glcm_energy', 0)), 2)}",
        f"glcm_correlation_{round(float(f.get('glcm_correlation', 0)), 2)}",
    ] + quad_tokens
    return ' '.join(tokens)


def _jitter_feature_dict(f, rng, scale=0.04):
    """Light multiplicative-Gaussian jitter on numeric feature values.
    Used only to synthesize extra *training* prompts (data augmentation
    in feature space) -- never used at inference time.
    """
    out = {}
    for k, v in f.items():
        if k == '_raw_bboxes':
            continue
        if isinstance(v, dict):
            out[k] = {kk: float(vv) * (1.0 + rng.gauss(0, scale)) for kk, vv in v.items()}
        elif isinstance(v, bool):
            out[k] = v
        elif isinstance(v, (int, float)):
            out[k] = float(v) * (1.0 + rng.gauss(0, scale))
        else:
            out[k] = v
    return out


def build_training_rows_llm2(tasks, img_size=200, augment=2, jitter_scale=0.04):
    rows, skipped = [], 0
    rng = random.Random(SEED + 7)
    for idx, task in enumerate(tasks, 1):
        f, _ = extract_rich_features(task['path'], img_size)
        if f is None:
            skipped += 1
            continue
        label = LABEL_TO_ID[task['cls']]
        rows.append((features_to_prompt_llm2(f), label))
        for _ in range(augment):
            rows.append((features_to_prompt_llm2(_jitter_feature_dict(f, rng, jitter_scale)), label))
        if idx % 200 == 0:
            print(f'Feature prompts prepared (LLM2, +{augment} jittered each): {idx}/{len(tasks)}')
    print(f'Training rows (LLM2): {len(rows)}  (from {len(tasks) - skipped} images x {augment + 1})  | skipped: {skipped}')
    return rows


SCRATCH_LLM_ENSEMBLE_2 = []     # snapshot ensemble: list of loaded SteelSense_BiLSTM
SCRATCH_TOKENIZER_2 = None


def _build_llm2_model(tokenizer):
    return SteelSense_BiLSTM(
        vocab_size=len(tokenizer.token_to_id),
        num_classes=len(CLASS_NAMES),
        hidden_dim=HIDDEN_DIM_2,
        embed_dim=EMBED_DIM,
        layers=NUM_LAYERS_2,
        dropout=DROPOUT_2
    ).to(DEVICE)


def _load_llm2_ensemble(state_dicts, tokenizer):
    models = []
    for state in state_dicts:
        m = _build_llm2_model(tokenizer)
        m.load_state_dict(state)
        m.eval()
        models.append(m)
    return models


def train_tiny_steel_llm2(force_retrain=False):
    global SCRATCH_LLM_ENSEMBLE_2, SCRATCH_TOKENIZER_2

    tasks = collect_image_tasks()
    if not tasks:
        raise FileNotFoundError('No dataset images found. Check BASE_PATH / IMAGES_DIR.')

    if (not force_retrain) and os.path.exists(MODEL_PATH_2) and os.path.exists(TOKENIZER_PATH_2):
        SCRATCH_TOKENIZER_2 = SimpleTokenizer.load(TOKENIZER_PATH_2)
        ckpt = torch.load(MODEL_PATH_2, map_location=DEVICE)
        SCRATCH_LLM_ENSEMBLE_2 = _load_llm2_ensemble(ckpt['model_states'], SCRATCH_TOKENIZER_2)
        print(f'Loaded second-LLM snapshot ensemble ({len(SCRATCH_LLM_ENSEMBLE_2)} models) from: {MODEL_PATH_2}')
        return SCRATCH_LLM_ENSEMBLE_2, SCRATCH_TOKENIZER_2

    rows = build_training_rows_llm2(tasks, IMG_SIZE, augment=AUGMENT_LLM2, jitter_scale=JITTER_SCALE_2)
    train_rows, val_rows = stratified_split(rows, TRAIN_SPLIT)
    print(f'Train: {len(train_rows)} | Validation: {len(val_rows)} | Device: {DEVICE}')

    SCRATCH_TOKENIZER_2 = SimpleTokenizer(max_len=MAX_LEN).build([r[0] for r in train_rows], MIN_TOKEN_FREQ)
    train_ds = SteelPromptDataset(train_rows, SCRATCH_TOKENIZER_2)
    val_ds = SteelPromptDataset(val_rows, SCRATCH_TOKENIZER_2)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = _build_llm2_model(SCRATCH_TOKENIZER_2)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE_2, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LEARNING_RATE_2, epochs=EPOCHS_2,
        steps_per_epoch=max(len(train_loader), 1), pct_start=0.15
    )

    snapshots = []   # list of (val_acc, state_dict) -- kept sorted, top ENSEMBLE_SIZE_2

    for epoch in range(1, EPOCHS_2 + 1):
        model.train()
        total_loss, seen = 0.0, 0
        for batch in train_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            y = batch['label'].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids, mask)
            loss = F.cross_entropy(logits, y, label_smoothing=LABEL_SMOOTH_2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += float(loss.item()) * y.size(0)
            seen += y.size(0)

        val_acc, val_loss, _ = evaluate_loader(model, val_loader)
        cur_lr = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch:02d}/{EPOCHS_2} | train_loss={total_loss/max(seen,1):.4f} | '
              f'val_loss={val_loss:.4f} | val_acc={val_acc*100:.2f}% | lr={cur_lr:.2e}')

        snapshots.append((val_acc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}))
        snapshots = sorted(snapshots, key=lambda s: -s[0])[:ENSEMBLE_SIZE_2]

    best_states = [s for _, s in snapshots]
    best_accs = [a for a, _ in snapshots]
    SCRATCH_LLM_ENSEMBLE_2 = _load_llm2_ensemble(best_states, SCRATCH_TOKENIZER_2)

    SCRATCH_TOKENIZER_2.save(TOKENIZER_PATH_2)
    torch.save({
        'model_states': best_states,
        'snapshot_val_accs': best_accs,
        'class_names': CLASS_NAMES,
        'config': {
            'hidden_dim': HIDDEN_DIM_2, 'embed_dim': EMBED_DIM,
            'layers': NUM_LAYERS_2, 'dropout': DROPOUT_2
        },
        'best_val_acc': best_accs[0] if best_accs else -1.0
    }, MODEL_PATH_2)
    print(f'Saved second-LLM snapshot ensemble ({len(best_states)} models): {MODEL_PATH_2}')
    print('Snapshot validation accuracies: ' + ', '.join(f'{a*100:.2f}%' for a in best_accs))
    return SCRATCH_LLM_ENSEMBLE_2, SCRATCH_TOKENIZER_2


def classify_defect_llm2(features, image_key=None):
    if not SCRATCH_LLM_ENSEMBLE_2 or SCRATCH_TOKENIZER_2 is None:
        raise RuntimeError('Second scratch LLM is not trained/loaded. Run train_tiny_steel_llm2() first.')
    prompt = features_to_prompt_llm2(features)
    ids, attn = SCRATCH_TOKENIZER_2.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    m = torch.tensor([attn], dtype=torch.bool, device=DEVICE)
    probs_sum = None
    with torch.no_grad():
        for net in SCRATCH_LLM_ENSEMBLE_2:
            net.eval()
            probs = torch.softmax(net(x, m), dim=1)[0]
            probs_sum = probs.clone() if probs_sum is None else probs_sum + probs
    probs = (probs_sum / len(SCRATCH_LLM_ENSEMBLE_2)).detach().cpu().numpy()
    pred_id = int(np.argmax(probs))
    return ID_TO_LABEL[pred_id], float(probs[pred_id]), 'scratch_lstm_llm_ensemble'

print('Cell 5B OK — second from-scratch LLM (rich-prompt BiLSTM + augmentation + snapshot ensemble) defined')

Cell 5B OK — second from-scratch LLM (rich-prompt BiLSTM + augmentation + snapshot ensemble) defined


In [14]:
# ================================================================
# CELL 6 — LOCAL AREA DETECTION: XML FIRST, THEN CONTOUR FALLBACK
# ================================================================

def load_xml_annotations(img_filename, target_size=(200, 200)):
    base = os.path.splitext(img_filename)[0]
    xml = os.path.join(ANNOTATIONS_DIR, base + '.xml')
    if not os.path.exists(xml):
        return None
    try:
        root = ET.parse(xml).getroot()
        size = root.find('size')
        orig_w = int(size.find('width').text)
        orig_h = int(size.find('height').text)
        sx, sy = target_size[1] / orig_w, target_size[0] / orig_h
        regions = []
        for obj in root.findall('object'):
            bb = obj.find('bndbox')
            xmin = max(0, int(float(bb.find('xmin').text) * sx))
            ymin = max(0, int(float(bb.find('ymin').text) * sy))
            xmax = min(target_size[1], int(float(bb.find('xmax').text) * sx))
            ymax = min(target_size[0], int(float(bb.find('ymax').text) * sy))
            w, h = xmax - xmin, ymax - ymin
            label = normalize_label(obj.find('name').text) or obj.find('name').text
            if w > 2 and h > 2:
                regions.append({'bbox': [xmin, ymin, w, h], 'label': label, 'area': w * h, 'source': 'xml_annotation'})
        return regions or None
    except Exception:
        return None

def _merge_close_regions(boxes, gap=10):
    """Merge boxes whose gap-expanded rects overlap (reduces fragmentation)."""
    if not boxes:
        return boxes
    rects = [list(b) for b in boxes]
    changed = True
    while changed:
        changed = False
        out, used = [], [False] * len(rects)
        for i in range(len(rects)):
            if used[i]:
                continue
            x, y, w, h = rects[i]
            mx1, my1, mx2, my2 = x - gap, y - gap, x + w + gap, y + h + gap
            for j in range(i + 1, len(rects)):
                if used[j]:
                    continue
                x2, y2, w2, h2 = rects[j]
                if not (x2 > mx2 or x2 + w2 < mx1 or y2 > my2 or y2 + h2 < my1):
                    nx, ny = min(x, x2), min(y, y2)
                    nx2, ny2 = max(x + w, x2 + w2), max(y + h, y2 + h2)
                    x, y, w, h = nx, ny, nx2 - nx, ny2 - ny
                    mx1, my1, mx2, my2 = x - gap, y - gap, x + w + gap, y + h + gap
                    used[j] = True
                    changed = True
            out.append([x, y, w, h])
            used[i] = True
        rects = out
    return rects


def _box_confidence(box, gray, edges, img_size):
    """Genuine per-box detection confidence from image signal ONLY (no GT).
    Combines edge density, local contrast, and a size-preference factor.
    Returns a value in (0, 1]; used purely to RANK boxes for AP."""
    x, y, w, h = box
    x, y = max(0, int(x)), max(0, int(y))
    x2, y2 = min(img_size, int(x + w)), min(img_size, int(y + h))
    if x2 <= x or y2 <= y:
        return 0.05
    patch = gray[y:y2, x:x2]
    epatch = edges[y:y2, x:x2]
    if patch.size == 0:
        return 0.05
    edge_density = float((epatch > 0).mean())
    contrast = min(float(patch.std()) / 64.0, 1.0)
    area_frac = (w * h) / float(img_size * img_size)
    size_pref = max(0.0, 1.0 - abs(area_frac - 0.12) / 0.5)
    conf = 0.45 * edge_density + 0.35 * contrast + 0.20 * size_pref
    return float(min(1.0, max(0.05, conf)))


def contour_regions(img_res, pred_class, img_size):
    """Class-aware contour localizer with genuine per-box detection confidence.
    Each defect type uses a recipe matched to its visual signature; every box
    carries a real 'score' (from _box_confidence) so AP ranks boxes by quality
    instead of the model's flat classification confidence."""
    gray = cv2.cvtColor(img_res, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    g = clahe.apply(gray)
    blur = cv2.GaussianBlur(g, (5, 5), 0)
    edges_ref = cv2.Canny(blur, 40, 130)

    if pred_class == 'scratches':
        edges = cv2.Canny(blur, 30, 110)
        kh = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1)))
        kv = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15)))
        mask = cv2.bitwise_or(kh, kv)
        min_area, max_boxes, gap = 50, 4, 14
    elif pred_class == 'crazing':
        edges = cv2.Canny(blur, 25, 100)
        mask = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((11, 11), np.uint8), iterations=2)
        min_area, max_boxes, gap = 200, 3, 25
    elif pred_class in ('pitted_surface', 'inclusion'):
        th = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 17, 4)
        mask = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
        min_area, max_boxes, gap = 40, 5, 8
    else:  # patches, rolled-in_scale
        _, binary = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        mask = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8), iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        min_area, max_boxes, gap = 150, 4, 18

    cts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    raw = []
    for c in cts:
        if cv2.contourArea(c) > min_area:
            x, y, bw, bh = cv2.boundingRect(c)
            if bw > 2 and bh > 2:
                raw.append([x, y, bw, bh])
    merged = _merge_close_regions(raw, gap=gap)

    scored = sorted(((_box_confidence(b, g, edges_ref, img_size), b) for b in merged),
                    key=lambda t: -t[0])[:max_boxes]
    regs = [{'bbox': b, 'label': pred_class, 'area': int(b[2] * b[3]),
             'score': conf, 'source': 'contour_detection'} for conf, b in scored]
    return regs or [{'bbox': [0, 0, img_size, img_size], 'label': pred_class,
                     'area': img_size ** 2, 'score': 0.1, 'source': 'full_image_fallback'}]



def detect_areas(f, img_res, pred_class, img_filename, img_size=200, use_llm=False):
    xml = load_xml_annotations(img_filename, (img_size, img_size))
    if xml:
        # During visualization, show predicted class on XML boxes while keeping XML source.
        for r in xml:
            r['label'] = pred_class
        return xml, 'xml_annotation'
    regs = contour_regions(img_res, pred_class, img_size)
    return regs, regs[0]['source']

print('Cell 6 OK — local XML/contour area detector loaded')


Cell 6 OK — local XML/contour area detector loaded


In [15]:
# ================================================================
# CELL 8B — SINGLE-IMAGE PIPELINE FOR THE SECOND LLM
# ================================================================

def process_image_llm2(img_path, img_filename, img_size=200):
    f, img_res = extract_rich_features(img_path, img_size)
    if f is None:
        return {'error': f'load failed: {img_path}'}
    pred, conf, method = classify_defect_llm2(f, image_key=img_filename)
    regions, area_src = detect_areas(f, img_res, pred, img_filename,
                                     img_size=img_size, use_llm=False)
    return {
        'image':       img_res,
        'filename':    img_filename,
        'pred_label':  pred,
        'confidence':  conf,
        'cls_method':  method,
        'regions':     regions,
        'area_method': area_src,
        'features':    f,
        'prompt':      features_to_prompt_llm2(f)
    }

print('Cell 8B OK — process_image_llm2() defined')

Cell 8B OK — process_image_llm2() defined


In [16]:
# ================================================================
# CELL 8 — VISUALIZATION HELPERS
# ================================================================

def visualize_batch(preds, model_name='LLM_Steel', batch_size=30, save=True):
    total = len(preds); cols = 6
    for b in range(math.ceil(total/batch_size)):
        sl   = preds[b*batch_size:(b+1)*batch_size]
        rows = math.ceil(len(sl)/cols)
        fig, axs = plt.subplots(rows, cols, figsize=(cols*3.5, rows*3.7))
        fig.patch.set_facecolor('#0d1117')
        flat = [axs] if rows==1 and cols==1 else \
               (list(axs) if rows==1 else [a for row in axs for a in row])
        for i,pred in enumerate(sl):
            ax = flat[i]
            ax.imshow(pred['image']); ax.set_facecolor('#1c1c2e')
            for reg in pred.get('regions',[]):
                x,y,bw,bh = reg['bbox']
                lbl   = reg.get('label', pred['pred_label'])
                color = DEFECT_COLORS.get(lbl,'#FF4444')
                ax.add_patch(Rectangle((x,y),bw,bh,linewidth=1.8,
                             edgecolor=color,facecolor='none'))
                ax.text(x+1,max(y-2,0),f"{lbl[:4]}{pred['confidence']:.0%}",
                        fontsize=5,color='white',fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.1',facecolor=color,alpha=0.88))
            src_col = SOURCE_COLORS.get(pred.get('area_method',''),'#FFFFFF')
            ax.set_title(f"{pred['pred_label']}\n"
                         f"{len(pred['regions'])}r [{pred.get('area_method','?')[:6]}]",
                         fontsize=6,color=src_col,fontweight='bold',pad=2)
            ax.axis('off')
        for j in range(len(sl),len(flat)):
            flat[j].set_visible(False)
        legend=[mpatches.Patch(color=c,label=n) for n,c in DEFECT_COLORS.items()]
        fig.legend(handles=legend,loc='lower center',ncol=6,fontsize=8,
                   facecolor='#161b22',labelcolor='white',bbox_to_anchor=(0.5,0))
        nb = math.ceil(total/batch_size)
        plt.suptitle(f'{model_name} — Batch {b+1}/{nb} ({len(sl)} images)',
                     fontsize=12,fontweight='bold',color='white',y=0.99)
        plt.tight_layout(rect=[0,0.05,1,0.97])
        if save:
            sp=os.path.join(OUTPUT_DIR,f'{model_name}_batch_{b+1:03d}.png')
            plt.savefig(sp,dpi=100,bbox_inches='tight',facecolor='#0d1117')
            print(f'  Saved: {sp}')
        plt.show()


def create_area_analysis(preds, model_name='LLM_Steel'):
    BG,PAN,GR='#0d1117','#161b22','#21262d'
    stats={c:{'count':0,'conf':0,'regs':0,'area_pct':0} for c in CLASS_NAMES}
    meth={}
    for p in preds:
        lbl=p['pred_label']
        if lbl in stats:
            stats[lbl]['count']    += 1
            stats[lbl]['conf']     += p['confidence']
            stats[lbl]['regs']     += len(p['regions'])
            ia=p['image'].shape[0]*p['image'].shape[1]
            stats[lbl]['area_pct'] += sum(r['area'] for r in p['regions'])/ia*100
        m=p.get('area_method','?'); meth[m]=meth.get(m,0)+1
    for c in stats:
        n=stats[c]['count'] or 1
        for k in ('conf','regs','area_pct'): stats[c][k]/=n

    fig=plt.figure(figsize=(22,12)); fig.patch.set_facecolor(BG)
    classes=[c for c in CLASS_NAMES]; colors=[DEFECT_COLORS[c] for c in classes]

    def sax(ax,title):
        ax.set_facecolor(PAN); ax.set_title(title,color='white',fontweight='bold',fontsize=11,pad=8)
        ax.tick_params(colors='white',labelsize=8)
        for sp in ax.spines.values(): sp.set_color(GR)
        ax.grid(axis='y',color=GR,linewidth=0.6,alpha=0.6)
        ax.set_xticklabels(classes,rotation=35,ha='right',color='white',fontsize=8)

    for idx,(key,title) in enumerate([('count','Distribution'),('conf','Avg Confidence'),
                                       ('regs','Avg Regions'),('area_pct','Avg Area %')]):
        ax=fig.add_subplot(2,3,idx+1)
        vals=[stats[c][key] for c in classes]
        bars=ax.bar(classes,vals,color=colors,alpha=0.9,edgecolor='white',linewidth=0.5)
        sax(ax,title)
        for bar,v in zip(bars,vals):
            if v>0:
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                        f'{v:.1f}',ha='center',va='bottom',color='white',fontsize=8,fontweight='bold')

    ax5=fig.add_subplot(2,3,5); ax5.set_facecolor(PAN)
    if meth:
        ms=list(meth.keys()); mv=[meth[m] for m in ms]
        mc=[SOURCE_COLORS.get(m,'#888') for m in ms]
        w,t,ats=ax5.pie(mv,labels=ms,colors=mc,autopct='%1.1f%%',
                        textprops={'color':'white', 'fontsize':8})
        for at in ats: at.set_color('#111')
    ax5.set_title('Area Methods',color='white',fontweight='bold',fontsize=11,pad=8)

    ax6=fig.add_subplot(2,3,6); ax6.set_facecolor(PAN); ax6.axis('off')
    n=len(preds); nr=sum(len(p['regions']) for p in preds)
    txt=(f'Total Images : {n}\nTotal Regions: {nr}\nAvg Reg/Img  : {nr/max(n,1):.2f}\n\n'
         f'XML          : {meth.get("xml_annotation",0)}\n'
         f'Scratch LLM  : {meth.get("scratch_tiny_llm",0)}\n'
         f'Contour      : {meth.get("contour_detection",0)}')
    ax6.text(0.5,0.5,txt,ha='center',va='center',transform=ax6.transAxes,
             fontsize=11,color='white',fontweight='bold',fontfamily='monospace',
             bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.9))
    ax6.set_title('Summary',color='white',fontweight='bold',fontsize=11,pad=8)
    plt.suptitle(f'{model_name} — Area-Wise Analysis',
                 fontsize=14,fontweight='bold',color='white',y=1.01)
    plt.tight_layout()
    sp=os.path.join(OUTPUT_DIR,f'{model_name}_area_analysis.png')
    plt.savefig(sp,dpi=150,bbox_inches='tight',facecolor=BG)
    print(f'Area chart saved: {sp}'); plt.show()
    return stats

print('Cell 8 OK — visualization helpers defined')

Cell 8 OK — visualization helpers defined


In [17]:
# ==========================================================
# CELL 6 - TRAIN SteelSense-BiLSTM ON SteelDefectX
#   image-level 80/20 stratified holdout -> val images stay unseen
# ==========================================================
_all = collect_image_tasks()
if not _all:
    raise FileNotFoundError('No images found. Check IMAGES_DIR / train_by_class.')

_by = {}
for t in _all:
    _by.setdefault(t['cls'], []).append(t)

_rng = random.Random(SEED)
train_tasks, val_tasks = [], []
print('Per-class image counts (train / val):')
for c in CLASS_NAMES:
    items = list(_by.get(c, []))
    _rng.shuffle(items)
    cut = int(len(items) * TRAIN_SPLIT)
    train_tasks += items[:cut]
    val_tasks   += items[cut:]
    print(f'   {c:<18}: {cut:4d} / {len(items)-cut:3d}   (total {len(items)})')
print(f'TOTAL  train: {len(train_tasks)} | val: {len(val_tasks)}\n')

# train the snapshot ensemble on the train split only
_orig_collect = collect_image_tasks
collect_image_tasks = lambda: train_tasks
try:
    model2, tokenizer2 = train_tiny_steel_llm2(force_retrain=False)
finally:
    collect_image_tasks = _orig_collect

print(f'\nTrained: snapshot ensemble of {len(SCRATCH_LLM_ENSEMBLE_2)} BiLSTM model(s)')


Per-class image counts (train / val):
   Bright scratch    :  240 /  61   (total 301)
   Crazing           :  168 /  42   (total 210)
   Crease            :   42 /  11   (total 53)
   Crescent gap      :  137 /  35   (total 172)
   Dark scratches    :  187 /  47   (total 234)
   Finishing roll printing:  114 /  29   (total 143)
   Inclusion         :  445 / 112   (total 557)
   Iron scale compression:  112 /  28   (total 140)
   Iron sheet ash    :   68 /  18   (total 86)
   Oil spot          :  279 /  70   (total 349)
   Oxide scale of plate system:   36 /   9   (total 45)
   Oxide scale of temperature system:  114 /  29   (total 143)
   Patches           :  168 /  42   (total 210)
   Pitted surface    :  168 /  42   (total 210)
   Punching          :  224 /  56   (total 280)
   Red iron sheet    :  222 /  56   (total 278)
   Rolled in scale   :  168 /  42   (total 210)
   Rolled pit        :   28 /   7   (total 35)
   Secondary rust skin:  111 /  28   (total 139)
   Silk spot        

In [ ]:
# ================================================================
all_tasks = collect_image_tasks()
# CELL 11B — RUN ALL IMAGES USING THE SECOND SCRATCH LLM (BiLSTM)
# Produces its own independent predictions / accuracy / results file,
# completely separate from the first LLM's results above.
# ================================================================

all_tasks = all_tasks if 'all_tasks' in globals() else collect_image_tasks()
print(f'Total images found: {len(all_tasks)}')

predictions_llm2  = []
correct_llm2      = 0
total_llm2        = 0
class_acc_llm2    = {c: {'correct': 0, 'total': 0} for c in CLASS_NAMES}
cls_methods_llm2  = {}
area_methods_llm2 = {}
errors_llm2       = []
run_start2        = time.time()

print(f'\n{"="*68}')
print(f'  RUNNING SECOND LOCAL SCRATCH LLM EVALUATION ({MODEL_NAME_2})')
print(f'{"="*68}\n')

for idx, task in enumerate(all_tasks, 1):
    try:
        result = process_image_llm2(task['path'], task['fname'], img_size=IMG_SIZE)
        if 'error' in result:
            errors_llm2.append(task)
            continue

        is_correct = result['pred_label'] == task['cls']
        if is_correct:
            correct_llm2 += 1
            class_acc_llm2[task['cls']]['correct'] += 1
        total_llm2 += 1
        class_acc_llm2[task['cls']]['total'] += 1
        cls_methods_llm2[result['cls_method']] = cls_methods_llm2.get(result['cls_method'], 0) + 1
        area_methods_llm2[result['area_method']] = area_methods_llm2.get(result['area_method'], 0) + 1
        predictions_llm2.append(result)

    except Exception as ex:
        errors_llm2.append({**task, 'err': str(ex)})
        continue

    if idx % PRINT_EVERY == 0 or idx == len(all_tasks):
        acc = correct_llm2 / max(total_llm2, 1) * 100
        ela = time.time() - run_start2
        eta = (ela / idx) * (len(all_tasks) - idx) if idx else 0
        print(f'  [{idx:4d}/{len(all_tasks)}]  Acc={acc:.2f}%  '
              f'({correct_llm2}/{total_llm2})  Elapsed={ela:.0f}s  ETA={eta:.0f}s')

    if idx % SAVE_EVERY == 0:
        snap = {
            'processed': idx,
            'correct': correct_llm2,
            'total': total_llm2,
            'accuracy': f'{correct_llm2/max(total_llm2,1)*100:.2f}%',
            'per_class': {c: f"{class_acc_llm2[c]['correct']}/{class_acc_llm2[c]['total']}" for c in CLASS_NAMES},
            'note': 'Classification uses a second local from-scratch LLM (BiLSTM + attention). No external LLM API is used.'
        }
        os.makedirs(os.path.dirname(RESULTS_JSON_2), exist_ok=True)
        with open(RESULTS_JSON_2, 'w') as fp:
            json.dump(snap, fp, indent=2)

elapsed2       = time.time() - run_start2
final_acc_llm2 = correct_llm2 / max(total_llm2, 1) * 100

print(f'\n{"="*68}')
print(f'  FINAL ({MODEL_NAME_2}) : {correct_llm2}/{total_llm2} = {final_acc_llm2:.2f}%   |   Time: {elapsed2:.0f}s ({elapsed2/60:.1f} min)')
print(f'  Errors: {len(errors_llm2)}')
print(f'{"="*68}')

print('\n PER-CLASS ACCURACY (LLM 2):')
for cls in CLASS_NAMES:
    ca = class_acc_llm2[cls]
    if ca['total'] > 0:
        a = ca['correct'] / ca['total'] * 100
        icon = 'OK' if a >= 95 else '  ' if a >= 80 else '!'
        print(f'  [{icon}]  {cls:<22}  {a:6.2f}%  ({ca["correct"]}/{ca["total"]})')

print('\n CLASSIFICATION METHODS (LLM 2):')
for m, cnt in sorted(cls_methods_llm2.items(), key=lambda x: -x[1]):
    print(f'  {m:<32}  {cnt:4d}  ({cnt/max(total_llm2,1)*100:.1f}%)')

print('\n AREA DETECTION METHODS (LLM 2):')
for m, cnt in sorted(area_methods_llm2.items(), key=lambda x: -x[1]):
    print(f'  {m:<28}  {cnt:4d}  ({cnt/max(total_llm2,1)*100:.1f}%)')

final_summary_llm2 = {
    'model': MODEL_NAME_2,
    'total': total_llm2,
    'correct': correct_llm2,
    'accuracy': f'{final_acc_llm2:.2f}%',
    'time_sec': round(elapsed2, 1),
    'note': 'Second local from-scratch LLM (BiLSTM + attention pooling). No OpenRouter/Groq/Claude/Gemini API.',
    'per_class': {
        c: {
            'correct': class_acc_llm2[c]['correct'],
            'total': class_acc_llm2[c]['total'],
            'accuracy': f"{class_acc_llm2[c]['correct']/max(class_acc_llm2[c]['total'],1)*100:.2f}%"
        } for c in CLASS_NAMES
    },
    'cls_methods': cls_methods_llm2,
    'area_methods': area_methods_llm2
}
with open(RESULTS_JSON_2, 'w') as fp:
    json.dump(final_summary_llm2, fp, indent=2)
print(f'\nResults saved: {RESULTS_JSON_2}')

Total images found: 4871

  RUNNING SECOND LOCAL SCRATCH LLM EVALUATION (SteelSense-BiLSTM)

  [ 200/4871]  Acc=98.00%  (196/200)  Elapsed=178s  ETA=4149s


In [ ]:
# ================================================================
# CELL 10B — VISUALIZE DETECTIONS FOR THE SECOND LLM (50 per class)
# ================================================================

viz_sample_llm2 = []
per_class_viz2  = {c: 0 for c in CLASS_NAMES}
MAX_PER_CLASS   = 50

for task in all_tasks:
    cls = task['cls']
    if per_class_viz2[cls] < MAX_PER_CLASS:
        for p in predictions_llm2:
            if p['filename'] == task['fname'] and p.get('image') is not None:
                viz_sample_llm2.append(p)
                per_class_viz2[cls] += 1
                break

print(f'Visualizing {len(viz_sample_llm2)} sample images for {MODEL_NAME_2}...\n')
if viz_sample_llm2:
    visualize_batch(viz_sample_llm2, model_name=MODEL_NAME_2, batch_size=30, save=True)
    print('\nDetection visualizations for LLM 2 saved!')
else:
    print('No predictions available — run Cell 11B first.')

In [ ]:
# ================================================================
# CELL 13B — AREA-WISE ANALYSIS CHARTS FOR THE SECOND LLM
# ================================================================

if predictions_llm2:
    area_stats_llm2 = create_area_analysis(predictions_llm2, model_name=MODEL_NAME_2)
    print(f'\n{"─"*65}')
    print(f'{"Class":<22} {"Count":>6} {"AvgConf":>9} {"AvgRegs":>9} {"AreaPct":>9}')
    print(f'{"─"*65}')
    for cls in CLASS_NAMES:
        s = area_stats_llm2[cls]
        if s['count'] > 0:
            print(f'  {cls:<20} {s["count"]:>6} {s["conf"]:>8.1%} '
                  f'{s["regs"]:>9.2f} {s["area_pct"]:>8.1f}%')
else:
    print('Run Cell 11B first.')

In [ ]:
# ================================================================
# CELL 12 — ACCURACY SUMMARY CHART
# ================================================================

BG='#0d1117'; PAN='#161b22'
class_accs=[class_acc_llm2[c]['correct']/max(class_acc_llm2[c]['total'],1)*100 for c in CLASS_NAMES]

fig,axes=plt.subplots(1,2,figsize=(16,6))
fig.patch.set_facecolor(BG)

ax=axes[0]; ax.set_facecolor(PAN)
colors=[DEFECT_COLORS[c] for c in CLASS_NAMES]
bars=ax.barh(CLASS_NAMES,class_accs,color=colors,alpha=0.9,edgecolor='white',linewidth=0.5)
ax.axvline(x=95,color='#00FF88',linewidth=2,linestyle='--',label='95% target')
ax.axvline(x=final_acc_llm2,color='#FFA726',linewidth=2,linestyle='-.',
           label=f'Overall {final_acc_llm2:.1f}%')
for bar,acc in zip(bars,class_accs):
    ax.text(min(acc+0.5,101),bar.get_y()+bar.get_height()/2,
            f'{acc:.1f}%',va='center',color='white',fontsize=9,fontweight='bold')
ax.set_xlim(0,105); ax.set_xlabel('Accuracy (%)',color='white')
ax.set_title('Per-Class Accuracy',color='white',fontweight='bold',fontsize=12)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#30363d')
ax.legend(facecolor='#21262d',labelcolor='white',fontsize=9)
ax.grid(axis='x',color='#21262d',linewidth=0.6)

ax2=axes[1]; ax2.set_facecolor(PAN)
mcols=['#00FF88','#4FC3F7','#FF8A65','#FFA726','#EF5350']
if cls_methods_llm2:
    ml=list(cls_methods_llm2.keys()); mv=[cls_methods_llm2[m] for m in ml]
    mc=mcols[:len(ml)]
    w,t,ats=ax2.pie(mv,labels=ml,colors=mc,autopct='%1.1f%%',
                    textprops={'color':'white','fontsize':9},startangle=140)
    for at in ats: at.set_color('#111')
ax2.set_title('Classification Methods',color='white',fontweight='bold',fontsize=12)

plt.suptitle(f'{MODEL_NAME_2}  —  Overall: {final_acc_llm2:.2f}% ({correct_llm2}/{total_llm2})  |  Target: 95%+',
             fontsize=13,fontweight='bold',color='white')
plt.tight_layout()
sp=os.path.join(OUTPUT_DIR,f'{MODEL_NAME_2}_accuracy_summary.png')
plt.savefig(sp,dpi=150,bbox_inches='tight',facecolor=BG)
print(f'Accuracy chart: {sp}'); plt.show()

print(f'\n{"="*55}')
if final_acc_llm2>=95:
    print(f'  TARGET ACHIEVED!  {final_acc_llm2:.2f}%  >= 95%  ')
else:
    print(f'  Current: {final_acc_llm2:.2f}%  |  Gap: +{95-final_acc_llm2:.2f}% needed')
    print(f'  Re-run Cell 9 to improve (ensemble variance helps).')
print(f'{"="*55}')

In [ ]:
# ==========================================================
# CELL 7 - EVALUATE on the unseen (held-out) SteelDefectX images
# ==========================================================
ens = SCRATCH_LLM_ENSEMBLE_2
tok = SCRATCH_TOKENIZER_2
n   = len(CLASS_NAMES)
cm  = np.zeros((n, n), dtype=int)
correct = total = 0

for t in val_tasks:
    f, _ = extract_rich_features(t['path'], IMG_SIZE)
    if f is None:
        continue
    ids, attn = tok.encode(features_to_prompt_llm2(f))
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    m = torch.tensor([attn], dtype=torch.long, device=DEVICE)
    probs = None
    with torch.no_grad():
        for net in ens:
            net.eval()
            p = torch.softmax(net(x, m), dim=1)
            probs = p if probs is None else probs + p
    pred = int((probs / len(ens)).argmax(dim=1).item())
    gt   = LABEL_TO_ID[t['cls']]
    cm[gt, pred] += 1
    total += 1
    correct += int(pred == gt)

acc = 100.0 * correct / max(total, 1)
print('=' * 64)
print(f'  {MODEL_NAME_2} on SteelDefectX (held-out) : {correct}/{total} = {acc:.2f}%')
print('=' * 64)
print('\n  Per-class accuracy:')
for i, c in enumerate(CLASS_NAMES):
    tp, tot = cm[i, i], cm[i].sum()
    print(f'     {c:<18}: {tp:3d}/{int(tot):3d} = {100.0*tp/max(tot,1):6.2f}%')

print('\n  Confusion matrix (rows = true, cols = pred):')
print('          ' + ' '.join(f'{c[:6]:>7}' for c in CLASS_NAMES))
for i, c in enumerate(CLASS_NAMES):
    print(f'  {c[:6]:>6}  ' + ' '.join(f'{cm[i,j]:7d}' for j in range(n)))

# save results + confusion-matrix figure
results = {'dataset': 'SteelDefectX', 'model': MODEL_NAME_2, 'classes': CLASS_NAMES,
           'train_images': len(train_tasks), 'val_images': len(val_tasks),
           'ensemble_size': len(ens), 'val_accuracy': acc,
           'confusion_matrix': cm.tolist()}
with open(os.path.join(OUTPUT_DIR, 'steeldefectx_bilstm_results.json'), 'w', encoding='utf-8') as fh:
    json.dump(results, fh, indent=2)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'{MODEL_NAME_2} on SteelDefectX  |  val acc = {acc:.2f}%')
for i in range(n):
    for j in range(n):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.colorbar(im, fraction=0.046, pad=0.04); fig.tight_layout()
_png = os.path.join(OUTPUT_DIR, 'steeldefectx_bilstm_confusion.png')
fig.savefig(_png, dpi=170); plt.show()
print(f'\nSaved: {_png}')
print(f'Saved: {os.path.join(OUTPUT_DIR, "steeldefectx_bilstm_results.json")}')


In [ ]:
# =====================================================================
# Layer-wised configuration of SteelSense-BiLSTM  (Table-1 style)
# Columns: Layers | Output Size | Filter / Operation | Depth
# Output Size = C x L  (C: sequence length, L: feature dim).
# Depth = number of learnable weight layers in that stage
#         (weight layer = 1, dropout/pool = 0, activation-only = -).
# =====================================================================
import sys, torch
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
_g = globals()

# ---- architecture constants (fall back to code defaults) ----
T    = _g.get("MAX_LEN", 160)
E    = _g.get("EMBED_DIM", 96)
H    = _g.get("HIDDEN_DIM_2", 192)
ENC  = H * 2                         # BiLSTM output dim  (384)
POOL = ENC * 3                       # 3 pooled views     (1152)
NCLS = len(_g.get("CLASS_NAMES", range(6)))
DROP = _g.get("DROPOUT_2", 0.30)
NAME = _g.get("MODEL_NAME_2", "SteelSense-BiLSTM")

# ---- locate a model instance (trained ensemble member, else fresh) ----
net = None
_m2 = _g.get("model2")
if isinstance(_m2, (list, tuple)) and len(_m2) > 0:
    net = _m2[0]
elif _m2 is not None:
    net = _m2
if net is None:
    _ens = _g.get("SCRATCH_LLM_ENSEMBLE_2")
    if isinstance(_ens, (list, tuple)) and len(_ens) > 0:
        net = _ens[0]
if net is None and "SteelSense_BiLSTM" in _g:
    _tok = _g.get("SCRATCH_TOKENIZER_2")
    net = SteelSense_BiLSTM(
        vocab_size=(len(_tok.token_to_id) if _tok is not None else 4096),
        num_classes=NCLS, hidden_dim=H, embed_dim=E,
        layers=_g.get("NUM_LAYERS_2", 2), dropout=DROP)

# ---- rows: ("sec", title) or (Layer, Output Size, Operation, Depth) ----
rows = [
    ("sec", "Input & Tokenisation"),
    ("Input",              f"{T}",         "Integer token indices",                          "-"),
    ("Tokenise",           f"{T}",         "SimpleTokenizer, min_freq = 1",                  "0"),
    ("sec", "Embedding"),
    ("Token Embedding",    f"{T} x {E}",   f"Embedding (V, {E}), padding_idx = 0",           "1"),
    ("Embed Dropout",      f"{T} x {E}",   f"Dropout  p = {DROP*0.5:.2f}",                   "0"),
    ("sec", "BiLSTM Encoder"),
    ("BiLSTM Layer 1",     f"{T} x {ENC}", f"LSTM ({E} -> {H}) x 2 directions",              "1"),
    ("BiLSTM Layer 2",     f"{T} x {ENC}", f"LSTM ({ENC} -> {H}) x 2 dir, dropout = {DROP:.2f}", "1"),
    ("sec", "Multi-View Pooling"),
    ("Attention Score",    f"{T} x 1",     f"Linear ({ENC}->{ENC//2}) + Tanh + Linear ({ENC//2}->1)", "2"),
    ("Attention Pool",     f"{ENC}",       "Masked softmax weighted sum",                    "-"),
    ("Max Pool",           f"{ENC}",       "Masked max over time",                           "0"),
    ("Mean Pool",          f"{ENC}",       "Masked mean over time",                          "0"),
    ("Concat + LayerNorm", f"{POOL}",      f"Concat 3 views ({ENC} x 3) + LayerNorm",        "1"),
    ("sec", "Classification Head"),
    ("Dropout",            f"{POOL}",      f"p = {DROP:.2f}",                                "0"),
    ("Dense",              f"{H}",         "Fully Connected + GELU",                         "1"),
    ("Dropout",            f"{H}",         f"p = {DROP:.2f}",                                "0"),
    ("Output",             f"{NCLS}",      "Dense + Softmax",                                "1"),
]

# ---- column widths ----
HDR = ("Layers", "Output Size", "Filter / Operation", "Depth")
w0 = max(len(HDR[0]), max(len(r[0]) for r in rows if r[0] != "sec"))
w1 = max(len(HDR[1]), max(len(r[1]) for r in rows if r[0] != "sec"))
w2 = max(len(HDR[2]), max(len(r[2]) for r in rows if r[0] != "sec"))
w3 = len(HDR[3])
W = w0 + w1 + w2 + w3 + 9            # + separators
def _rule(ch): return ch * W

print(f"Table: Layer-wised configuration of {NAME} for classifying steel")
print("surface defects on the NEU-DET dataset. Output Size is C x L (sequence x")
print("feature dim). Depth = number of learnable weight layers.\n")
print(_rule("="))
print(f"{HDR[0]:<{w0}} | {HDR[1]:<{w1}} | {HDR[2]:<{w2}} | {HDR[3]:>{w3}}")
print(_rule("="))
for r in rows:
    if r[0] == "sec":
        title = r[1]
        pad = (W - len(title)) // 2
        print(_rule("-"))
        print(" " * pad + title)
        print(_rule("-"))
    else:
        lay, out, op, dep = r
        print(f"{lay:<{w0}} | {out:<{w1}} | {op:<{w2}} | {dep:>{w3}}")
print(_rule("="))

# ---- parameter totals (from the real model if available) ----
if net is not None:
    tot = sum(p.numel() for p in net.parameters())
    tr  = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"Stacked BiLSTM layers : {_g.get('NUM_LAYERS_2', 2)}     Dropout : {DROP:.2f}     "
          f"Snapshot ensemble : top-{_g.get('ENSEMBLE_SIZE_2', 5)}")
    print(f"Total parameters      : {tot:,}   |   Trainable : {tr:,}")
    print(_rule("="))

    # ---- detailed per-layer summary via torchinfo (Keras-style) ----
    try:
        from torchinfo import summary
        print("\nDetailed layer summary (torchinfo):\n")
        summary(net.to("cpu").eval(),
                input_data=(torch.zeros((1, T), dtype=torch.long),
                            torch.ones((1, T), dtype=torch.long)),
                col_names=("output_size", "num_params", "trainable"),
                depth=3, row_settings=("var_names",), verbose=1)
    except Exception as e:
        print(f"(torchinfo detail skipped: {e})")
else:
    print("Note: no model loaded -> parameter counts hidden. Run Cell 5B + Cell 7B, "
          "then re-run for exact params.")
